# 예제 05. 분류 모델 만들고 돌려보기
빅데이터프로그래밍 · 6주차

## 목표
- 모델 구조를 출력한다
- 임의의 입력을 넣어 출력 shape을 확인한다
- 학습을 돌려 손실과 정확도가 개선되는 것을 본다

5주차 학습 루프의 `model` 자리에 오늘 만든 신경망을 넣습니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 — 세 덩어리로 나뉜 2차원 점
직선 하나로는 나눌 수 없는 배치입니다.


In [ ]:
def make_blobs(n_per=200):
    centers = torch.tensor([[0.0, 2.0], [-2.0, -1.0], [2.0, -1.0]])
    X, y = [], []
    for i, c in enumerate(centers):
        X.append(torch.randn(n_per, 2) * 0.7 + c)
        y.append(torch.full((n_per,), i))
    return torch.cat(X), torch.cat(y)


X, y = make_blobs()
print("X:", X.shape, "y:", y.shape, "클래스:", y.unique().tolist())

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=12)
plt.title("data"); plt.show()


In [ ]:
perm = torch.randperm(len(X))
X, y = X[perm], y[perm]

n_train = int(len(X) * 0.8)
train_loader = DataLoader(TensorDataset(X[:n_train], y[:n_train]), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X[n_train:], y[n_train:]), batch_size=32, shuffle=False)

print("학습:", n_train, "검증:", len(X) - n_train, "batch 수:", len(train_loader))


## 2. 모델 작성


In [ ]:
class Classifier(nn.Module):
    def __init__(self, in_f=2, h=32, n_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(in_f, h)
        self.fc2 = nn.Linear(h, h)
        self.fc3 = nn.Linear(h, n_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


model = Classifier().to(device)
print(model)
print("파라미터:", sum(p.numel() for p in model.parameters()))


## 3. 임의의 입력으로 출력 shape 확인
학습 전에 반드시 한 번 해 봅니다. shape이 틀리면 여기서 걸립니다.


In [ ]:
dummy = torch.randn(4, 2).to(device)          # 데이터 4개
out = model(dummy)

print("입력 :", tuple(dummy.shape))
print("출력 :", tuple(out.shape))              # (4, 3) — 클래스 3개
print("확률 :", torch.softmax(out, dim=1)[0].data.round(decimals=3))


## 4. 학습


In [ ]:
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

def accuracy(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            correct += (model(bx).argmax(dim=1) == by).sum().item()
            total += by.numel()
    return correct / total


history = []
for epoch in range(40):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)

        loss = loss_fn(model(bx), by)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item() * bx.shape[0]

    train_loss = total_loss / n_train
    val_acc = accuracy(val_loader)
    history.append((train_loss, val_acc))

    if epoch % 8 == 0 or epoch == 39:
        print(f"epoch {epoch:3d}  loss {train_loss:.4f}  val acc {val_acc:.3f}")


## 5. 손실과 정확도 곡선


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([h[0] for h in history]); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot([h[1] for h in history]); ax[1].set_title("val accuracy"); ax[1].set_xlabel("epoch")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. 모델이 나눈 경계 그려보기
은닉층과 ReLU가 있어서 경계가 직선이 아닙니다.


In [ ]:
import numpy as np

xx, yy = torch.meshgrid(torch.linspace(-5, 5, 200), torch.linspace(-5, 5, 200), indexing="xy")
grid = torch.stack([xx.flatten(), yy.flatten()], dim=1).to(device)

model.eval()
with torch.no_grad():
    zz = model(grid).argmax(dim=1).cpu().reshape(xx.shape)

plt.figure(figsize=(5.5, 5))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="viridis")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=10)
plt.title("decision boundary"); plt.show()


## 7. 은닉층을 빼면 — 직선 경계만 나옵니다


In [ ]:
linear_only = nn.Linear(2, 3).to(device)
opt2 = torch.optim.Adam(linear_only.parameters(), lr=0.01)

for _ in range(40):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        loss = loss_fn(linear_only(bx), by)
        opt2.zero_grad(); loss.backward(); opt2.step()

with torch.no_grad():
    zz2 = linear_only(grid).argmax(dim=1).cpu().reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, z, t in [(ax[0], zz, "은닉층 2개"), (ax[1], zz2, "선형 1층")]:
    a.contourf(xx, yy, z, alpha=0.25, cmap="viridis")
    a.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=8)
    a.set_title(t)
plt.tight_layout(); plt.show()


## 직접 해보기
1. 은닉층 크기를 4로 줄이면 경계가 어떻게 달라지나요?
2. ReLU를 Sigmoid로 바꿔 정확도를 비교하세요.
3. 클래스를 4개로 늘려 데이터를 만들고 모델을 고치세요.


In [ ]:
# 여기에 작성하세요
